# BERTopic hyperparameter tuning

Searches UMAP and HDBSCAN settings for BERTopic with Hyperopt (Tree of Parzen Estimators), using precomputed document embeddings. The objective maximises topic coherence (NPMI) plus topic diversity.

The search reported in the paper was run on the 1955–1970 subset with `Alibaba-NLP/gte-base-en-v1.5` document embeddings, computed on the fully preprocessed text (stopwords removed, lemmatised; the *word-level* variant). To tune another embedding model or period, change the settings cell below.

In [ ]:
# !pip install hyperopt

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

sys.path.append('../../evaluation')

from evaluation import Trainer
from data import DataLoader

### Settings

In [ ]:
# Embedding model and its precomputed document embeddings
EMBEDDING_MODEL = "Alibaba-NLP/gte-base-en-v1.5"
EMBEDDINGS_FILE = "../../../datasets/compare_study_dataset/aggregation/word_embeddings_octis_Alibaba-NLP_gte-base-en-v1.5.npy"

# Dataset in OCTIS format (1955-1970 subset, fully preprocessed text)
dataset, custom = "../../../datasets/compare_study_dataset/aggregation/word_embedding_data_8192", True

# Number of hyperparameter configurations to evaluate
MAX_EVALS = 300

# Output
RESULTS_DIR = "../../evaluation_results/hyperparameter_tuning/"
RUN_NAME = "words_gte"
os.makedirs(RESULTS_DIR, exist_ok=True)

### Load data and embeddings

In [ ]:
data_loader = DataLoader(dataset)
_, timestamps = data_loader.load_docs()
data = data_loader.load_octis(custom)
data = [" ".join(words) for words in data.get_corpus()]

model = SentenceTransformer(EMBEDDING_MODEL, trust_remote_code=True)
embeddings = np.load(EMBEDDINGS_FILE)

len(data), embeddings.shape

### Search space

In [ ]:
param_space = {
    'min_topic_size': hp.choice('min_topic_size', np.arange(10, 50, dtype=int)),
    'n_components': hp.choice('n_components', np.arange(2, 10, dtype=int)),
    'n_neighbors': hp.choice('n_neighbors', np.arange(8, 20, dtype=int)),
    'min_cluster_size': hp.choice('min_cluster_size', np.arange(20, 50, dtype=int)),
    'min_samples':hp.choice('min_samples', np.arange(20, 50, dtype=int)),
}

### Objective function and search

In [ ]:
params_results = []

def objective_hyperopt(params):
    print("Parameters: ", params)
    params = {
            "embedding_model": model,
            "min_topic_size": params['min_topic_size'],
            "umap_model": UMAP(angular_rp_forest=True, metric='cosine', n_components=params['n_components'], n_neighbors=params['n_neighbors'], n_jobs=1, random_state=42, verbose=False),
            "hdbscan_model": HDBSCAN(gen_min_span_tree=True, min_cluster_size=params['min_cluster_size'], min_samples=params['min_samples'], prediction_data=True),
            "verbose": False
        }
    trainer = Trainer(dataset=dataset,
                          model_name="BERTopic",
                          params=params,
                          bt_embeddings=embeddings,
                          custom_dataset=custom,
                          custom_model=None,
                          verbose=False)

    results, _ = trainer.train()
    params_results.append((params, results))
    print("Score:", results[0]["Scores"])
    # Hyperopt minimises the loss, so the sum of coherence and diversity is negated
    return {'loss': -(results[0]["Scores"]['npmi'] + results[0]["Scores"]['diversity']), 'status': STATUS_OK}

trials = Trials()
best = fmin(objective_hyperopt, param_space, algo=tpe.suggest, max_evals=MAX_EVALS, trials=trials)

# hp.choice returns the index of each chosen value, not the value itself
print(f"best hyperparameters (indices into the search space): {best}")

### Save the trials

In [ ]:
with open(os.path.join(RESULTS_DIR, f"hyperopt_trials_{RUN_NAME}.txt"), "w") as file:
    for i, trial in enumerate(trials.trials):
        hyperparameters = trial['misc']['vals']
        objective_value = trial['result']['loss']
        file.write(f"Trial {i+1}: Hyperparameters: {hyperparameters}, Objective Value: {objective_value}\n")

In [ ]:
# One row per trial: hyperparameters, scores, runtime and number of topics
rows = []
for params, results in params_results:
    result = results[0]
    rows.append({
        "min_topic_size": params["min_topic_size"],
        "n_components": params["umap_model"].n_components,
        "n_neighbors": params["umap_model"].n_neighbors,
        "min_cluster_size": params["hdbscan_model"].min_cluster_size,
        "min_samples": params["hdbscan_model"].min_samples,
        "npmi": result["Scores"]["npmi"],
        "diversity": result["Scores"]["diversity"],
        "Computation Time": result["Computation Time"],
        "Number of Topics": result["Number of Topics"],
    })
results_df = pd.DataFrame(rows)

results_df.to_csv(os.path.join(RESULTS_DIR, f"params_results_{RUN_NAME}.csv"), index=False, sep="|")
results_df

### Inspect the trials and select a configuration

The configurations with the highest objective value tend to be degenerate: they produce only a handful of topics, which gives a topic diversity close to 1. The final configuration is therefore selected among the trials that produce a large number of topics, by topic coherence.

In [ ]:
results_df.plot.scatter(x='Number of Topics', y='npmi')
results_df.plot.scatter(x='Number of Topics', y='diversity')

In [ ]:
results_df[results_df['Number of Topics'] > 140].sort_values(by='npmi', ascending=False)